# Step 1: Conflict inventory on clean SinD trajectories

A **TTC conflict** needs all of: minimum (exact, rectangle-based) TTC below the threshold, at
least 3 frames below it, and an **actual** footprint gap of at most 1 m in the 3 s after the
minimum-TTC moment. Pairs closing slower than 1 m/s are not scanned. **PET conflicts** are
crossing pairs below the PET threshold.

Both must also pass the **plausibility checks**: neither road user is static (track extent
under 3 m: parked cars, stopped tricycles), and their footprints overlap for at most 0.3 s
(longer overlap means a box or heading error in the data, or someone getting in or out of a
vehicle). `exclusions.csv` reports how many candidates each rule removed.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # project root

import pandas as pd
from sensors_noise.pipeline import run_conflict_check
from sensors_noise.conflicts import summarise, gap_sensitivity

DATA_ROOT = Path(r"PATH\TO\sind")
OUT_DIR = Path.cwd().parent / "results" / "01_conflict_check"

## Full run (a few minutes; Changchun is the slowest)

In [ ]:
res = run_conflict_check(DATA_ROOT, OUT_DIR)
pair_ttc, pair_pet = res["pair_ttc"], res["pair_pet"]

## What the plausibility checks removed

In [ ]:
res["exclusions"]

## Per-site detail

In [ ]:
res["by_recording"]

## Sensitivity to the persistence filter

If counts drop sharply from 1 to 3 frames, many candidates are one-frame spikes.

In [ ]:
for k in (1, 3, 5):
    print(f"min_frames = {k}")
    print(summarise(pair_ttc, pair_pet, by=("group",), min_frames=k).to_string(), "\n")

## Sensitivity to the minimum relative speed

This needs a rescan, so each setting takes as long as the full run.

In [ ]:
speed_sens = {}
for v in (0.5, 2.0):
    r = run_conflict_check(DATA_ROOT, OUT_DIR.parent / f"01_speed_{v}", min_rel_speed=v,
                           make_plots=False, verbose=False)
    speed_sens[f"{v} m/s"] = r["groups"].loc["All recordings"]
speed_sens["1.0 m/s"] = res["groups"].loc["All recordings"]
pd.concat(speed_sens, axis=1)

## Distributions

In [ ]:
from IPython.display import Image
Image(filename=str(OUT_DIR / "distributions.png"))

## Closest interactions (for visual sanity checks)

In [ ]:
pair_ttc[pair_ttc["min_gap"] <= 1.0].nsmallest(15, "min_ttc")

In [ ]:
pair_pet.nsmallest(15, "pet")

## Step 1b: visual check of flagged events

Plots 3 events per category into `results/01_conflict_check/events/`. **Left:** both trajectories
(dot = start of the window), footprints at the reference moment (solid), the constant-velocity
projection to the predicted collision (dashed, TTC events) or the conflict point (star, PET
events); grey dots are all other trajectories, which outline the lanes. **Right:** actual footprint
gap and exact TTC over time.

What to look for: do opposite-direction cases look like left turns across oncoming traffic or
like straight passes? Do Chongqing pedestrian cases look like people walking along the road edge?
Do PET values near 0 show trajectories that look physically implausible (tracking errors)?

In [ ]:
from sensors_noise.events import plot_selected_events
events = plot_selected_events(DATA_ROOT, OUT_DIR, n=3)
events[["category", "recording", "uid_a", "uid_b", "min_ttc", "min_gap", "pet"]]

In [ ]:
from IPython.display import Image, display
for f in events["figure"]:
    if f:
        display(Image(filename=f))